# Computation of fixation probabilities and mean segregation times

This notebook computes the probability that the two diffusions
$$dp_t = s_0p_t(1-p_t)[h+p_t(1-2h)]dt + \frac{1}{\sqrt{2}}\sqrt{p_t(1-p_t)}\,dB_t$$ and
$$dq_t = hs_0q_t(1-q_t)dt + \sqrt{2}\sqrt{q_t(1-q_t)}\,dB_t$$
reach the absorbing state $1$ before the absorbing state $0$.
It also computes the expected time these diffusions spend away from the absorbing states, when initiated at a non-trivial value in $(0,1)$.

It allows one to pick a desired population size $N$, which impacts the initial frequencies in the way that $p_0=1/(2N)$ and $q_0=2/N$, and the selection parameters $h$ and $s_0$.

$(p_t)_{t\geq0}$ represents the frequency of a mutation appearing on an autosome, while $(q_t)_{t\geq0}$ represents the frequency of a mutation appearing on a Y chromosome, both in a diploid population whose dynamics are described in Section 2 of [Article Link].

The following code approximates the quantities defined in Appendix A.3. of [https://www.biorxiv.org/content/10.64898/2026.04.01.715871v1].
     





In [2]:
from scipy import integrate
from scipy import special
import numpy as np

# Population parameters
pop_size = 10000   # diploid population of size N

x_init_q = 2 / pop_size       # initial frequency for q_t
x_init_p = 1 / (2 * pop_size) # initial frequency for p_t

In [3]:
# Selection coefficients

S= [10]
#S = [-5, -1, -0.1, 0.1, 1, 5, 10, 20, 50, 100]

H=[1/2]
#H = [-2, -1, -0.5, -0.1, 0.1, 0.5, 1, 2]

In [4]:
def expuv(x, u, v):
  """ Auxiliary function"""
  return np.exp(u * x**2 - v * x)

In [5]:
def fy(s):
    """Fixation probability for a Y-linked mutation """
    return (1 - np.exp(-s * x_init_q)) / (1 - np.exp(-s))


def gy1(xi, s):
    """Left integrand for the mean segregation time on Y """
    return (1 - np.exp(-xi * s)) / (xi * (1 - xi) * s * np.exp(-xi * s))


def gy2(xi, s):
    """Right integrand for the mean segregation time on Y """
    return (np.exp(-xi * s) - np.exp(-s)) / (xi * (1 - xi) * s * np.exp(-xi * s))

In [6]:
def H1(x, u, v):
    """Left integrand for the autosomal mean segregation time """
    return integrate.quad(expuv, 0, x, args=(u, v))[0] / (x * (1 - x) * expuv(x, u, v))


def H2(x, u, v):
    """Right integrand for the autosomal mean segregation time """
    return integrate.quad(expuv, x, 1, args=(u, v))[0] / (x * (1 - x) * expuv(x, u, v))

## Y-linked locus: fixation probability and mean segregation time

In [7]:
probability_Y = np.zeros((len(S), len(H)))  # fixation probabilities
time_Y   = np.zeros((len(S), len(H)))  # mean segregation times

for i, s in enumerate(S):
    for j, h in enumerate(H):

        s_eff = s * h  # effective selection coefficient on Y

        # Fixation probability
        probability_Y[i, j] = fy(s_eff)

        # Mean segregation time
        u1 = integrate.quad(gy1, 0, x_init_q, args=(s_eff,), limit=3000)[0]
        v1 = integrate.quad(gy2, x_init_q, 1, args=(s_eff,), limit=3000)[0]

        time_Y[i, j] = (1 - probability_Y[i, j]) * u1 + probability_Y[i, j] * v1

print("Fixation probabilities (Y):")
print(probability_Y)
print("\nMean segregation times (Y):")
print(time_Y)

Fixation probabilities (Y):
[[0.00100628]]

Mean segregation times (Y):
[[0.0022912]]


## Autosomal locus: fixation probability and mean segregation time

In [8]:
probability_autos = np.zeros((len(S), len(H)))  # fixation probabilities

for i, s in enumerate(S):
    for j, h in enumerate(H):

        # Parameters of expuv function
        u = -2 * s * (1 - 2 * h)
        v = -4 * s * h

        # Fixation probability
        numerator   = integrate.quad(expuv, 0, x_init_p, args=(u, v))[0]
        denominator = integrate.quad(expuv, 0, 1, args=(u, v))[0]
        probability_autos[i, j] = numerator / denominator

print("Fixation probabilities (autosome):")
print(probability_autos)

Fixation probabilities (autosome):
[[2.06218455e-12]]


In [9]:
time_autos = np.zeros((len(S), len(H)))  # mean segregation times

for i, s in enumerate(S):
    for j, h in enumerate(H):

        u = -2 * s * (1 - 2 * h)
        v = -4 * s * h

        p = probability_autos[i, j]


        left  = 4 * (1 - p) * integrate.quad(H1, 0, x_init_p, args=(u, v), limit=5000)[0]
        right = 4 * p        * integrate.quad(H2, x_init_p, 1, args=(u, v), limit=5000)[0]

        time_autos[i, j] = left + right

print("Mean segregation times (autosome):")
print(time_autos)

Mean segregation times (autosome):
[[0.00147745]]
